# 기상청 API 기반 현재 날씨 정보 조회 코드

이 코드는 기상청 단기예보 조회서비스 중 `초단기실황조회 getUltraSrtNcst` API를 사용하여  
사용자가 선택한 도시의 현재 날씨 정보를 가져오는 코드입니다.

조회 대상 기상 요소는 다음과 같습니다.

- T1H: 기온
- REH: 습도
- PTY: 강수형태
- RN1: 1시간 강수량
- WSD: 풍속

도시는 Streamlit의 `selectbox`와 연결하기 쉽도록 딕셔너리 형태로 미리 저장합니다.

In [1]:
import os
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

## 1. 도시별 기상청 격자 좌표 정의

기상청 API는 위도와 경도가 아니라 `nx`, `ny` 형태의 격자 좌표를 사용합니다.

따라서 사용자가 도시명을 선택하면, 해당 도시의 기상청 격자 좌표를 반환할 수 있도록  
도시별 좌표를 딕셔너리로 저장합니다.

이 구조는 Streamlit에서 `selectbox`로 도시를 선택할 때 사용하기 좋습니다.

In [2]:
CITY_GRID = {
    "서울": {"nx": 60, "ny": 127},
    "부산": {"nx": 98, "ny": 76},
    "대구": {"nx": 89, "ny": 90},
    "인천": {"nx": 55, "ny": 124},
    "광주": {"nx": 58, "ny": 74},
    "대전": {"nx": 67, "ny": 100},
    "울산": {"nx": 102, "ny": 84},
    "세종": {"nx": 66, "ny": 103},
    "경기": {"nx": 60, "ny": 120},
    "강원": {"nx": 73, "ny": 134},
    "충북": {"nx": 69, "ny": 107},
    "충남": {"nx": 55, "ny": 107},
    "전북": {"nx": 63, "ny": 89},
    "전남": {"nx": 51, "ny": 67},
    "경북": {"nx": 87, "ny": 106},
    "경남": {"nx": 91, "ny": 77},
    "제주": {"nx": 52, "ny": 38},
}

## 2. 코드값 및 기상 요소 정보 정의

기상청 API에서 일부 값은 숫자 코드로 반환됩니다.

예를 들어 `PTY`는 강수형태를 의미하며,  
`0`, `1`, `2` 같은 코드값으로 반환됩니다.

사용자에게 그대로 보여주기보다는  
`강수없음`, `비`, `눈`처럼 텍스트로 변환하는 것이 좋습니다.

In [3]:
PTY_MAP = {
    "0": "강수없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울/눈날림",
    "7": "눈날림",
}

CATEGORY_INFO = {
    "T1H": {"name": "기온", "unit": "℃"},
    "REH": {"name": "습도", "unit": "%"},
    "PTY": {"name": "강수형태", "unit": "코드값"},
    "RN1": {"name": "1시간 강수량", "unit": "mm"},
    "WSD": {"name": "풍속", "unit": "m/s"},
}

## 3. 도시명으로 격자 좌표 반환 함수

사용자가 입력하거나 선택한 도시명을 기준으로  
해당 도시의 `nx`, `ny` 좌표를 반환하는 함수입니다.

지원하지 않는 도시명이 들어오면 오류 메시지와 함께  
사용 가능한 도시 목록을 출력합니다.

In [4]:
def get_city_grid(city: str) -> tuple[int, int]:
    city = city.strip()

    if city not in CITY_GRID:
        available = ", ".join(CITY_GRID.keys())
        raise ValueError(
            f"지원하지 않는 도시입니다: {city}\n"
            f"사용 가능 도시: {available}"
        )

    return CITY_GRID[city]["nx"], CITY_GRID[city]["ny"]

## 4. 초단기실황 조회 기준 시간 계산 함수

초단기실황 자료는 매시 정각 기준으로 생성됩니다.

다만 API는 보통 매시각 10분 이후에 안정적으로 조회할 수 있으므로,  
현재 시간이 10분 이전이면 이전 시간의 자료를 조회하도록 처리합니다.

예시는 다음과 같습니다.

- 14시 09분이면 13시 자료 조회
- 14시 10분이면 14시 자료 조회

In [5]:
def get_ultra_srt_ncst_base_time(now: datetime | None = None) -> tuple[str, str]:
    if now is None:
        now = datetime.now(ZoneInfo("Asia/Seoul"))

    if now.minute < 10:
        now = now - timedelta(hours=1)

    base_date = now.strftime("%Y%m%d")
    base_time = now.strftime("%H00")

    return base_date, base_time

## 5. 결측값 처리 함수

기상청 API에서는 관측값이 없거나 비정상적인 경우  
특정 값이 결측값처럼 반환될 수 있습니다.

이 함수는 다음 값을 결측으로 처리합니다.

- None
- 빈 문자열
- "-"
- "null"
- 900 이상
- -900 이하

In [6]:
def is_missing_value(value) -> bool:
    if value in [None, "", "-", "null"]:
        return True

    try:
        value_float = float(value)
        return value_float >= 900 or value_float <= -900
    except ValueError:
        return False

## 6. 1시간 강수량 RN1 텍스트 변환 함수

`RN1`은 1시간 강수량을 의미합니다.

API에서는 숫자 형태의 mm 값으로 반환되지만,  
음악 추천 키워드 생성이나 화면 출력에서는 텍스트 범주가 더 사용하기 좋습니다.

예를 들어 다음과 같이 변환합니다.

- 0 → 강수없음
- 1mm 미만 → 1mm 미만
- 3mm 미만 → 약한 비
- 15mm 미만 → 보통 비
- 30mm 미만 → 강한 비

In [7]:
def map_rn1_to_text(value) -> str:
    if is_missing_value(value):
        return "결측"

    try:
        rain = float(value)
    except ValueError:
        return "알 수 없음"

    if rain == 0:
        return "강수없음"
    elif rain < 1.0:
        return "1mm 미만"
    elif rain < 3.0:
        return "약한 비"
    elif rain < 15.0:
        return "보통 비"
    elif rain < 30.0:
        return "강한 비"
    elif rain < 50.0:
        return "매우 강한 비"
    else:
        return "극심한 비"

## 7. 기상청 초단기실황 API 조회 함수

이 함수는 실제로 기상청 API에 요청을 보내고,  
현재 날씨 정보를 딕셔너리 형태로 반환합니다.

입력값은 다음과 같습니다.

- city: 도시명
- api_key: 기상청 API 인증키

반환값은 다음 정보를 포함합니다.

- 도시명
- 격자 좌표
- 조회 기준 날짜
- 조회 기준 시간
- 기온, 습도, 강수형태, 강수량, 풍속

In [8]:
def get_current_weather(city: str, api_key: str | None = None) -> dict:
    if api_key is None:
        api_key = os.getenv("KMA_API_KEY")

    if not api_key:
        raise ValueError(
            "기상청 API 키가 없습니다. "
            "api_key를 직접 넣거나 KMA_API_KEY 환경변수를 설정하세요."
        )

    nx, ny = get_city_grid(city)
    base_date, base_time = get_ultra_srt_ncst_base_time()

    url = (
        "https://apihub.kma.go.kr/api/typ02/openApi/"
        "VilageFcstInfoService_2.0/getUltraSrtNcst"
    )

    params = {
        "pageNo": 1,
        "numOfRows": 1000,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": nx,
        "ny": ny,
        "authKey": api_key,
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    header = data.get("response", {}).get("header", {})
    result_code = str(header.get("resultCode"))
    result_msg = header.get("resultMsg")

    if result_code not in ["0", "00"]:
        raise RuntimeError(f"기상청 API 오류: {result_code}, {result_msg}")

    items = data["response"]["body"]["items"]["item"]

    target_categories = ["T1H", "REH", "PTY", "RN1", "WSD"]
    raw_weather = {}

    for item in items:
        category = item.get("category")
        value = item.get("obsrValue")

        if category in target_categories:
            raw_weather[category] = value

    result = {
        "city": city,
        "nx": nx,
        "ny": ny,
        "base_date": base_date,
        "base_time": base_time,
        "weather": {}
    }

    for category in target_categories:
        value = raw_weather.get(category)

        if is_missing_value(value):
            parsed_value = None
            text_value = "결측"

        elif category == "PTY":
            parsed_value = value
            text_value = PTY_MAP.get(str(value), "알 수 없음")

        elif category == "RN1":
            parsed_value = float(value)
            text_value = map_rn1_to_text(value)

        else:
            parsed_value = float(value)
            text_value = f"{parsed_value}{CATEGORY_INFO[category]['unit']}"

        result["weather"][category] = {
            "name": CATEGORY_INFO[category]["name"],
            "value": parsed_value,
            "unit": CATEGORY_INFO[category]["unit"],
            "text": text_value,
        }

    return result

## 8. 실행 예시

아래 코드는 서울의 현재 날씨 정보를 조회하는 예시입니다.

실제 실행 시에는 `"발급받은_기상청_API_KEY"` 부분에  
기상청 API 인증키를 입력하면 됩니다.

In [ ]:
KMA_API_KEY = ""

weather = get_current_weather("서울", KMA_API_KEY)

weather

{'city': '서울',
 'nx': 60,
 'ny': 127,
 'base_date': '20260525',
 'base_time': '1600',
 'weather': {'T1H': {'name': '기온',
   'value': 29.8,
   'unit': '℃',
   'text': '29.8℃'},
  'REH': {'name': '습도', 'value': 47.0, 'unit': '%', 'text': '47.0%'},
  'PTY': {'name': '강수형태', 'value': '0', 'unit': '코드값', 'text': '강수없음'},
  'RN1': {'name': '1시간 강수량', 'value': 0.0, 'unit': 'mm', 'text': '강수없음'},
  'WSD': {'name': '풍속', 'value': 2.2, 'unit': 'm/s', 'text': '2.2m/s'}}}

## 9. 필요한 값만 출력하는 예시

반환된 딕셔너리에서 필요한 값만 따로 출력할 수도 있습니다.

Streamlit 화면에 보여줄 때는 이 형태가 더 편합니다.

In [10]:
print("도시:", weather["city"])
print("기준 날짜:", weather["base_date"])
print("기준 시간:", weather["base_time"])

print("기온:", weather["weather"]["T1H"]["text"])
print("습도:", weather["weather"]["REH"]["text"])
print("강수형태:", weather["weather"]["PTY"]["text"])
print("1시간 강수량:", weather["weather"]["RN1"]["text"])
print("풍속:", weather["weather"]["WSD"]["text"])

도시: 서울
기준 날짜: 20260525
기준 시간: 1600
기온: 29.8℃
습도: 47.0%
강수형태: 강수없음
1시간 강수량: 강수없음
풍속: 2.2m/s
